# DATASET DE VALIDACIÓN PARA STM32Cube.AI / ST Edge AI


 Genera un .npy por cada modelo TFLite.

 Entrada física:
   $X_{0}.csv -> [y, \alpha, y_{dot}, \alpha_{dot}]$

 Luego:
   1. Selección reproducible de N muestras
   2. Normalización EXACTA usada por la red
   3. Conversión al dtype esperado por cada TFLite
   4. Guardado en formato .npy

 Forma final:

       (N_VALIDACION, 4)



In [5]:

import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
import pickle

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

N_VALIDACION = 200
SEED = 100


# ------------------------------------------------------------
# Ajusta esta ruta a tu X0.csv
# ------------------------------------------------------------

PATH_X0 = Path(
    "FinalAMPCPendulumDataset/X0.csv"
)


# ------------------------------------------------------------
# Carpeta donde ya tienes los modelos TFLite
#
# En tu notebook probablemente OUTPUT_TFLITE ya existe.
# ------------------------------------------------------------

OUTPUT_TFLITE = Path("modelos_cuantizados")

OUTPUT_VALIDACION = Path(
    "datasets_validacion_stm32cubeai"
)

OUTPUT_VALIDACION.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 75)
print("GENERACIÓN DATASET VALIDACIÓN STM32Cube.AI")
print("=" * 75)


# ============================================================
# 2. CARGAR X0.csv
# ============================================================

X0 = pd.read_csv(
    PATH_X0,
    header=None
).values.astype(np.float32)


print("\nDataset original:")

print(
    "Shape:",
    X0.shape
)

print(
    "dtype:",
    X0.dtype
)


if X0.shape[1] != 4:

    raise ValueError(
        f"X0 debería tener 4 estados, "
        f"pero tiene shape={X0.shape}"
    )


if len(X0) < N_VALIDACION:

    raise ValueError(
        f"Se solicitaron {N_VALIDACION} muestras, "
        f"pero X0 solo contiene {len(X0)}."
    )


# ============================================================
# 3. SELECCIÓN REPRODUCIBLE
# ============================================================

rng = np.random.default_rng(
    SEED
)


indices = rng.choice(
    len(X0),
    size=N_VALIDACION,
    replace=False
)


X_val_fisico = X0[
    indices
].copy()


print("\nMuestras seleccionadas:")

print(
    "Shape:",
    X_val_fisico.shape
)


# ============================================================
# 4. NORMALIZACIÓN
# ============================================================
#
# Usamos EXACTAMENTE la misma función que utiliza
# el controlador original.
#
# Se asume que ya tienes definidos:
#
#   normalize(...)
#   normalization["x"]
#
# en el notebook.
#
# ============================================================

MODEL_DIR = Path(r"model")
NORMALIZATION_PATH = MODEL_DIR / "normalization_reconstructed.pkl"

for p in [NORMALIZATION_PATH]:
    print(f"{p.name:16s} -> {'OK' if p.exists() else 'NO ENCONTRADO'}")

if NORMALIZATION_PATH.exists():
    with open(NORMALIZATION_PATH, "rb") as f:
        normalization = pickle.load(f)
    NORMALIZATION_SOURCE = "normalization.p ORIGINAL"
else:
    mu_x, scale_x = stats_media_maxabs(X0_PATH, [0,1,2,3])
    mu_u, scale_u = stats_media_maxabs(U_PATH, [0])
    normalization = {"x": (mu_x, scale_x), "u": (mu_u, scale_u)}
    NORMALIZATION_SOURCE = "RECONSTRUIDA DESDE DATASET"

print("Fuente:", NORMALIZATION_SOURCE)
print("x:", normalization["x"])
print("u:", normalization["u"])


def normalize(data, p):
    return (data - p[0]) / p[1]


X_val_norm = np.asarray(
    [
        normalize(
            x,
            normalization["x"]
        )
        for x in X_val_fisico
    ],
    dtype=np.float32
)


print("\nDataset normalizado:")

print(
    "Shape:",
    X_val_norm.shape
)

print(
    "dtype:",
    X_val_norm.dtype
)

print(
    "mínimo:",
    X_val_norm.min(axis=0)
)

print(
    "máximo:",
    X_val_norm.max(axis=0)
)


# ============================================================
# 5. GUARDAR TAMBIÉN ESTADOS FÍSICOS
# ============================================================
#
# Este archivo NO se entrega al modelo.
# Solo sirve para trazabilidad.
#
# ============================================================

np.save(
    OUTPUT_VALIDACION /
    "X_validation_fisico.npy",
    X_val_fisico
)


# ============================================================
# 6. FUNCIÓN:
#    CONVERTIR FLOAT32 NORMALIZADO AL DTYPE DEL TFLITE
# ============================================================

def preparar_input_tflite(
    path_modelo,
    X_float
):

    interpreter = tf.lite.Interpreter(
        model_path=str(path_modelo)
    )

    interpreter.allocate_tensors()


    input_detail = (
        interpreter
        .get_input_details()[0]
    )


    dtype = input_detail[
        "dtype"
    ]


    qparams = input_detail[
        "quantization_parameters"
    ]


    scales = np.asarray(
        qparams["scales"],
        dtype=np.float64
    )


    zero_points = np.asarray(
        qparams["zero_points"],
        dtype=np.int64
    )


    # ========================================================
    # FLOAT32
    # ========================================================

    if dtype == np.float32:

        X_modelo = (
            X_float
            .astype(np.float32)
        )


    # ========================================================
    # FLOAT16
    # ========================================================

    elif dtype == np.float16:

        X_modelo = (
            X_float
            .astype(np.float16)
        )


    # ========================================================
    # INPUT CUANTIZADO
    # ========================================================

    elif dtype in [
        np.int8,
        np.uint8,
        np.int16
    ]:

        if len(scales) == 0:

            raise RuntimeError(
                f"{path_modelo.name}: "
                "input entero sin escala de cuantización."
            )


        # ----------------------------------------------------
        # Caso habitual:
        # quantización per-tensor del INPUT
        # ----------------------------------------------------

        if len(scales) == 1:

            scale = float(
                scales[0]
            )

            zero_point = int(
                zero_points[0]
            )


            if scale == 0:

                raise RuntimeError(
                    f"{path_modelo.name}: "
                    "scale del input = 0."
                )


            X_q = np.round(
                X_float / scale
                +
                zero_point
            )


        # ----------------------------------------------------
        # Caso raro:
        # input per-axis
        # ----------------------------------------------------

        else:

            quantized_dimension = int(
                qparams[
                    "quantized_dimension"
                ]
            )


            shape_broadcast = [
                1
            ] * X_float.ndim


            shape_broadcast[
                quantized_dimension
            ] = len(scales)


            scale = scales.reshape(
                shape_broadcast
            )

            zero_point = (
                zero_points
                .reshape(
                    shape_broadcast
                )
            )


            X_q = np.round(
                X_float / scale
                +
                zero_point
            )


        # ----------------------------------------------------
        # Saturación al rango representable
        # ----------------------------------------------------

        info = np.iinfo(
            dtype
        )


        X_modelo = np.clip(
            X_q,
            info.min,
            info.max
        ).astype(
            dtype
        )


    else:

        raise TypeError(
            f"Dtype no soportado: {dtype}"
        )


    return (
        X_modelo,
        input_detail
    )


# ============================================================
# 7. BUSCAR TODOS LOS TFLITE
# ============================================================

modelos_tflite = sorted(
    OUTPUT_TFLITE.glob(
        "*.tflite"
    )
)


if len(modelos_tflite) == 0:

    raise FileNotFoundError(
        f"No se encontraron modelos .tflite "
        f"en {OUTPUT_TFLITE}"
    )


print("\n" + "=" * 75)
print("MODELOS ENCONTRADOS")
print("=" * 75)


for path in modelos_tflite:

    print(
        " -",
        path.name
    )


# ============================================================
# 8. CREAR .npy PARA CADA MODELO
# ============================================================

resultados = []


for path_modelo in modelos_tflite:

    X_st, input_detail = (
        preparar_input_tflite(
            path_modelo,
            X_val_norm
        )
    )


    # --------------------------------------------------------
    # Nombre archivo
    # --------------------------------------------------------

    nombre_salida = (
        f"validation_"
        f"{path_modelo.stem}.npy"
    )


    path_salida = (
        OUTPUT_VALIDACION /
        nombre_salida
    )


    np.save(
        path_salida,
        X_st
    )


    # --------------------------------------------------------
    # Validar archivo recién guardado
    # --------------------------------------------------------

    X_check = np.load(
        path_salida
    )


    assert (
        X_check.shape
        ==
        X_st.shape
    )

    assert (
        X_check.dtype
        ==
        X_st.dtype
    )


    # --------------------------------------------------------
    # Información de cuantización
    # --------------------------------------------------------

    quant = input_detail[
        "quantization"
    ]


    resultados.append(
        {
            "Modelo":
                path_modelo.name,

            "Archivo_validacion":
                nombre_salida,

            "Shape":
                str(X_st.shape),

            "Dtype":
                str(X_st.dtype),

            "Input_scale":
                quant[0],

            "Input_zero_point":
                quant[1],

            "Min":
                float(
                    np.min(X_st)
                ),

            "Max":
                float(
                    np.max(X_st)
                ),
        }
    )


# ============================================================
# 9. RESUMEN
# ============================================================

df_validacion_st = pd.DataFrame(
    resultados
)


print("\n" + "=" * 75)
print("DATASETS GENERADOS")
print("=" * 75)


display(
    df_validacion_st
)


print(
    "\nCarpeta:",
    OUTPUT_VALIDACION.resolve()
)


print(
    "\nNúmero de muestras:",
    N_VALIDACION
)


print(
    "\n✓ Datasets listos para STM32Cube.AI"
)

GENERACIÓN DATASET VALIDACIÓN STM32Cube.AI

Dataset original:
Shape: (732110, 4)
dtype: float32

Muestras seleccionadas:
Shape: (200, 4)
normalization_reconstructed.pkl -> OK
Fuente: normalization.p ORIGINAL
x: (array([ 6.71002547e-05,  4.41114071e-03,  1.87137396e-04, -3.78032260e-03]), array([0.34999895, 6.28318017, 1.19999886, 9.99999844]))
u: (array([-0.00770264]), array([9.00000009]))

Dataset normalizado:
Shape: (200, 4)
dtype: float32
mínimo: [-0.9795592  -0.9981702  -0.99643624 -0.98832726]
máximo: [0.9935882  0.99476093 0.98494047 0.9914308 ]

MODELOS ENCONTRADOS
 - _temp_base_debug.tflite
 - AMPC_CartPole_MixedPrecision.tflite
 - AMPC_CartPole_PTQ_INT16x8.tflite
 - AMPC_CartPole_PTQ_INT8_PerChannel.tflite
 - AMPC_CartPole_PTQ_INT8_PerTensor.tflite
 - AMPC_CartPole_QAT_INT8.tflite

DATASETS GENERADOS


C:\Users\Alonso\.conda\envs\CNN\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,Modelo,Archivo_validacion,Shape,Dtype,Input_scale,Input_zero_point,Min,Max
0,_temp_base_debug.tflite,validation__temp_base_debug.npy,"(200, 4)",float32,0.000000,0,-0.99817,0.994761
1,AMPC_CartPole_MixedPrecision.tflite,validation_AMPC_CartPole_MixedPrecision.npy,"(200, 4)",float32,0.000000,0,-0.99817,0.994761
2,AMPC_CartPole_PTQ_INT16x8.tflite,validation_AMPC_CartPole_PTQ_INT16x8.npy,"(200, 4)",float32,0.000000,0,-0.99817,0.994761
3,AMPC_CartPole_PTQ_INT8_PerChannel.tflite,validation_AMPC_CartPole_PTQ_INT8_PerChannel.npy,"(200, 4)",int8,0.007843,-1,-128.00000,126.000000
4,AMPC_CartPole_PTQ_INT8_PerTensor.tflite,validation_AMPC_CartPole_PTQ_INT8_PerTensor.npy,"(200, 4)",int8,0.007843,-1,-128.00000,126.000000
5,AMPC_CartPole_QAT_INT8.tflite,validation_AMPC_CartPole_QAT_INT8.npy,"(200, 4)",int8,0.007847,0,-127.00000,127.000000



Carpeta: C:\Users\Alonso\Jupyter\Tesis\Test_Modelos_Cuantizacion\Adaptative MPC\datasets_validacion_stm32cubeai

Número de muestras: 200

✓ Datasets listos para STM32Cube.AI
